# Métriques RAGAS — runs v3_clean

Calcul des métriques RAGAS sur les runs générés par `eval_v3clean_3configs.ipynb`.

**Métriques :**
- `faithfulness` — réponse fidèle au contexte ?
- `answer_relevancy` — réponse pertinente par rapport à la question ?
- `context_precision` — items pertinents bien classés ?
- `context_recall` — contexte couvre la gold_answer ?
- `answer_correctness` — réponse correspond à gold_answer ?
- `answer_similarity` — similarité sémantique réponse vs gold_answer

**LLM juge :** OpenAI `gpt-4o-mini`

In [ ]:
import os, sys, json, time
import pandas as pd
import psycopg
from psycopg.rows import dict_row
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
DSN = os.getenv("TUNNEL_DSN") or os.getenv("SCALINGO_POSTGRESQL_URL") or os.getenv("PG_DSN")

with psycopg.connect(DSN, row_factory=dict_row) as conn:
    rows = conn.execute("""
        SELECT config_name, COUNT(*) as cnt,
               SUM(CASE WHEN metrics IS NOT NULL AND metrics != '{}'::jsonb THEN 1 ELSE 0 END) as has_metrics
        FROM goldset_runs
        WHERE config_name LIKE 'v3clean_%'
        GROUP BY config_name ORDER BY config_name
    """).fetchall()

print("Runs v3clean_* :")
for r in rows:
    print(f"  {r['config_name']}: {r['cnt']} runs, {r['has_metrics']} with metrics")

In [ ]:
# RAGAS setup
from ragas import evaluate as ragas_evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness,
    answer_similarity,
)
from datasets import Dataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

RAGAS_MODEL = "gpt-4o-mini"
ragas_llm = ChatOpenAI(model=RAGAS_MODEL, temperature=0)
ragas_embeddings = OpenAIEmbeddings()

METRICS_NO_GOLD = [faithfulness, answer_relevancy]
METRICS_WITH_GOLD = [faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness, answer_similarity]

print(f"RAGAS ready — judge: {RAGAS_MODEL}")
print(f"  OpenAI key: {'OK' if os.getenv('OPENAI_API_KEY') else 'MISSING'}")

## Configuration

In [ ]:
# Configs to evaluate (set to the ones that have runs)
CONFIGS_TO_EVAL = [
    "v3clean_prod",
    "v3clean_plus_test",
    "v3clean_test_only",
]

SKIP_ALREADY_COMPUTED = True
BATCH_SIZE = 50

## Fonctions utilitaires

In [ ]:
def load_runs(config_name: str, skip_computed: bool = True) -> pd.DataFrame:
    """Load runs for a config, optionally skipping already-computed ones."""
    where = "gr.config_name = %s AND gr.response IS NOT NULL AND gr.retrieved_context IS NOT NULL"
    if skip_computed:
        where += " AND (gr.metrics IS NULL OR gr.metrics = '{}'::jsonb)"
    sql = f"""
        SELECT gr.id as run_id, gr.question_id, gq.question, gq.gold_answer,
               gq.goldset_name, gr.config_name, gr.response, gr.retrieved_context, gr.metrics
        FROM goldset_runs gr
        JOIN goldset_questions_v2 gq ON gr.question_id = gq.id
        WHERE {where}
        ORDER BY gr.id
    """
    with psycopg.connect(DSN, row_factory=dict_row) as conn:
        rows = conn.execute(sql, (config_name,)).fetchall()
    return pd.DataFrame(rows)


def extract_context_texts(retrieved_context) -> list[str]:
    """Extract text strings from retrieved_context JSONB for RAGAS."""
    if not retrieved_context:
        return [""]
    if isinstance(retrieved_context, str):
        try:
            retrieved_context = json.loads(retrieved_context)
        except Exception:
            return [""]
    if not isinstance(retrieved_context, list):
        return [""]

    texts = []
    for item in retrieved_context:
        text = item.get("text", "") or item.get("content", "")
        source = item.get("source", "")
        title = item.get("title", "")
        if text:
            header = f"[{source}] {title}" if source else title
            texts.append(f"{header}\n{text}" if header else text)
    return texts if texts else [""]


def has_gold(row) -> bool:
    gold = row.get("gold_answer")
    return bool(gold and str(gold).strip())


def prepare_dataset(df: pd.DataFrame) -> Dataset:
    data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
    for _, row in df.iterrows():
        data["question"].append(row["question"])
        data["answer"].append(row["response"] or "")
        data["contexts"].append(extract_context_texts(row["retrieved_context"]))
        gold = row.get("gold_answer")
        data["ground_truth"].append(str(gold) if gold and str(gold).strip() else "")
    return Dataset.from_dict(data)


def save_metrics(run_id: int, metrics: dict):
    """Merge metrics into goldset_runs.metrics JSONB (preserves existing keys)."""
    with psycopg.connect(DSN) as conn:
        conn.execute(
            "UPDATE goldset_runs SET metrics = COALESCE(metrics, '{}'::jsonb) || %s::jsonb WHERE id = %s",
            (json.dumps(metrics), run_id),
        )
        conn.commit()


print("Utility functions ready")

## Debug : inspecter ce que RAGAS voit

Vérifier la qualité des données avant de lancer l'évaluation ($$$).

In [ ]:
# Charger les runs de la première config à évaluer
cfg_check = CONFIGS_TO_EVAL[0]
df_check = load_runs(cfg_check, skip_computed=SKIP_ALREADY_COMPUTED)
print(f"{cfg_check}: {len(df_check)} runs à évaluer")

if len(df_check) > 0:
    mask_gold = df_check.apply(has_gold, axis=1)
    print(f"  Avec gold_answer: {mask_gold.sum()}/{len(df_check)}")

    for _, row in df_check.head(3).iterrows():
        print(f"\n{'='*60}")
        print(f"Q{row['question_id']}: {row['question'][:80]}...")
        print(f"  Gold: {'YES' if has_gold(row) else 'NO'}")

        ctx = row["retrieved_context"]
        if isinstance(ctx, str):
            try:
                ctx = json.loads(ctx)
            except Exception:
                ctx = []

        print(f"  Raw context: {len(ctx) if ctx else 0} items")
        if ctx:
            for j, item in enumerate(ctx[:3]):
                text = item.get("text", "") or item.get("content", "")
                print(f"    [{j}] {item.get('source','')[:20]} | {item.get('title','')[:40]} | {len(text)} chars | score={item.get('score','?')}")

        ragas_ctx = extract_context_texts(row["retrieved_context"])
        total_chars = sum(len(c) for c in ragas_ctx)
        print(f"  RAGAS sees: {len(ragas_ctx)} strings, {total_chars} total chars")
        print(f"  Response: {len(row['response'] or '')} chars")

## Test sur 10 questions

Vérifier que RAGAS fonctionne et estimer le coût/temps avant le full run.

In [ ]:
TEST_N = 10
df_test = df_check.head(TEST_N)
if len(df_test) == 0:
    print("Aucun run à tester")
else:
    mask = df_test.apply(has_gold, axis=1)
    metrics_to_use = METRICS_WITH_GOLD if mask.any() else METRICS_NO_GOLD
    ds = prepare_dataset(df_test)

    t0 = time.time()
    result = ragas_evaluate(ds, metrics=metrics_to_use, llm=ragas_llm, embeddings=ragas_embeddings)
    elapsed = time.time() - t0
    scores = result.to_pandas()

    print(f"Test {TEST_N} questions en {elapsed:.0f}s ({elapsed/TEST_N:.1f}s/question)")
    metric_cols = ["faithfulness", "answer_relevancy", "context_precision",
                   "context_recall", "answer_correctness", "answer_similarity"]
    for col in metric_cols:
        if col in scores.columns:
            vals = scores[col].dropna()
            if len(vals) > 0:
                print(f"  {col:25s}: mean={vals.mean():.3f} | min={vals.min():.3f} | max={vals.max():.3f}")

    remaining = len(df_check) - TEST_N
    est_min = remaining * (elapsed / TEST_N) / 60
    est_cost = remaining * 0.001  # rough estimate
    print(f"\nEstimation full run ({remaining} restants): ~{est_min:.0f} min, ~${est_cost:.2f}")

## Boucle d'évaluation RAGAS

In [ ]:
def evaluate_config(config_name: str) -> pd.DataFrame:
    """Run RAGAS on all pending runs for a config."""
    df = load_runs(config_name, skip_computed=SKIP_ALREADY_COMPUTED)
    if len(df) == 0:
        print(f"  {config_name}: no runs to evaluate")
        return pd.DataFrame()

    mask_gold = df.apply(has_gold, axis=1)
    print(f"  {config_name}: {len(df)} runs ({mask_gold.sum()} with gold)")

    all_scores = []
    t0 = time.time()

    for label, group_df, metrics in [
        ("WITH gold", df[mask_gold].reset_index(drop=True), METRICS_WITH_GOLD),
        ("NO gold", df[~mask_gold].reset_index(drop=True), METRICS_NO_GOLD),
    ]:
        if len(group_df) == 0:
            continue

        n_batches = (len(group_df) + BATCH_SIZE - 1) // BATCH_SIZE
        print(f"    {label}: {len(group_df)} runs, {n_batches} batches")

        for b in range(n_batches):
            start = b * BATCH_SIZE
            end = min(start + BATCH_SIZE, len(group_df))
            batch_df = group_df.iloc[start:end]
            print(f"      Batch {b+1}/{n_batches} [{start+1}-{end}]...", end=" ")

            try:
                ds = prepare_dataset(batch_df)
                tb = time.time()
                result = ragas_evaluate(ds, metrics=metrics, llm=ragas_llm, embeddings=ragas_embeddings)
                scores = result.to_pandas()
                elapsed_b = time.time() - tb

                metric_cols = ["faithfulness", "answer_relevancy", "context_precision",
                               "context_recall", "answer_correctness", "answer_similarity"]
                for i_row in range(len(batch_df)):
                    run_id = batch_df.iloc[i_row]["run_id"]
                    m = {"computed_at": datetime.now().isoformat(), "judge_model": RAGAS_MODEL, "method": "ragas"}
                    for col in metric_cols:
                        if col in scores.columns:
                            val = scores.iloc[i_row][col]
                            if pd.notna(val):
                                m[col] = round(float(val), 4)
                    save_metrics(run_id, m)

                avgs = {col: scores[col].dropna().mean() for col in metric_cols if col in scores.columns and scores[col].dropna().shape[0] > 0}
                summary = ", ".join(f"{k}={v:.3f}" for k, v in avgs.items())
                print(f"{elapsed_b:.0f}s — {summary}")
                all_scores.append(scores)

            except Exception as e:
                print(f"FAILED: {str(e)[:100]}")

    elapsed = time.time() - t0
    print(f"  Done in {elapsed/60:.1f} min")
    return pd.concat(all_scores, ignore_index=True) if all_scores else pd.DataFrame()

In [ ]:
# Summary of what needs evaluation
for cfg in CONFIGS_TO_EVAL:
    df = load_runs(cfg, skip_computed=SKIP_ALREADY_COMPUTED)
    if len(df) > 0:
        gold_pct = df.apply(has_gold, axis=1).mean() * 100
        print(f"  {cfg}: {len(df)} runs needing metrics ({gold_pct:.0f}% with gold)")
    else:
        print(f"  {cfg}: all metrics already computed (or no runs)")

In [ ]:
# Run RAGAS for each config
all_results = {}
for cfg_name in CONFIGS_TO_EVAL:
    print(f"\n{'='*60}")
    scores = evaluate_config(cfg_name)
    if len(scores) > 0:
        all_results[cfg_name] = scores

## Résumé comparatif

In [ ]:
# Load all computed metrics from DB for comparison
comparison = []
for cfg_name in CONFIGS_TO_EVAL:
    df = load_runs(cfg_name, skip_computed=False)
    if len(df) == 0:
        continue

    metrics_vals = {}
    metric_cols = ["faithfulness", "answer_relevancy", "context_precision",
                   "context_recall", "answer_correctness", "answer_similarity"]
    for _, row in df.iterrows():
        m = row.get("metrics")
        if isinstance(m, str):
            m = json.loads(m)
        if not isinstance(m, dict):
            continue
        for col in metric_cols:
            if col in m and m[col] is not None:
                metrics_vals.setdefault(col, []).append(float(m[col]))

    row_data = {"config": cfg_name, "n_runs": len(df)}
    for col in metric_cols:
        vals = metrics_vals.get(col, [])
        row_data[col] = f"{sum(vals)/len(vals):.3f}" if vals else "-"
    comparison.append(row_data)

df_cmp = pd.DataFrame(comparison)
print("\nComparaison des 3 configurations v3_clean :")
print()
print(df_cmp.to_string(index=False))

## Retry des runs sans métriques

In [ ]:
for cfg_name in CONFIGS_TO_EVAL:
    df_retry = load_runs(cfg_name, skip_computed=True)
    if len(df_retry) > 0:
        print(f"{cfg_name}: {len(df_retry)} runs still need metrics — retrying...")
        evaluate_config(cfg_name)
    else:
        print(f"{cfg_name}: all done")

## Export CSV

In [ ]:
# Export toutes les métriques v3clean en CSV
metric_cols = ["faithfulness", "answer_relevancy", "context_precision",
               "context_recall", "answer_correctness", "answer_similarity"]
export_rows = []
for cfg_name in CONFIGS_TO_EVAL:
    df = load_runs(cfg_name, skip_computed=False)
    for _, row in df.iterrows():
        m = row.get("metrics")
        if isinstance(m, str):
            try:
                m = json.loads(m)
            except Exception:
                m = {}
        if not isinstance(m, dict):
            m = {}
        r = {
            "config": cfg_name,
            "run_id": row["run_id"],
            "question_id": row["question_id"],
            "question": row["question"],
            "goldset_name": row.get("goldset_name", ""),
        }
        for col in metric_cols:
            r[col] = m.get(col)
        export_rows.append(r)

df_export = pd.DataFrame(export_rows)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
csv_path = f"metrics_v3clean_{timestamp}.csv"
df_export.to_csv(csv_path, index=False)
n_with = df_export[metric_cols].notna().any(axis=1).sum()
print(f"Exported {len(df_export)} rows ({n_with} with metrics) to {csv_path}")

---

**Next step:** Visualiser les résultats dans `08_Eval_Comparison.py` (les 3 configs `v3clean_*` apparaissent automatiquement).